# JobSpy Job Scraper

This notebook uses [JobSpy](https://github.com/speedyapply/JobSpy) to scrape jobs from LinkedIn, Indeed, Glassdoor, Google, ZipRecruiter & more.

**Usage:** Modify the search parameters below and run all cells to fetch jobs.

In [ ]:
# Install JobSpy if not already installed
!pip install -U python-jobspy -q

In [ ]:
import pandas as pd
from jobspy import scrape_jobs
import json
from IPython.display import display, HTML, JSON
from datetime import datetime

## Search Parameters

Customize these parameters to search for jobs:

In [ ]:
# Search configuration
SEARCH_TERM = "data analyst"  # Job title/keywords
LOCATION = "Remote"  # Location (e.g., "San Francisco, CA", "Remote", "India")
SITES = ["indeed", "linkedin", "zip_recruiter", "google"]  # Job boards to search
RESULTS_WANTED = 50  # Number of results per site
HOURS_OLD = 72  # Only jobs posted in last N hours
COUNTRY = "USA"  # For Indeed/Glassdoor (USA, India, UK, etc.)
IS_REMOTE = True  # Filter for remote jobs only

## Scrape Jobs

This will fetch jobs from the specified job boards:

In [ ]:
print(f"🔍 Searching for '{SEARCH_TERM}' jobs in '{LOCATION}'...")
print(f"📊 Sites: {', '.join(SITES)}")
print(f"⏰ Jobs posted in last {HOURS_OLD} hours")
print("\n⏳ This may take 1-3 minutes...\n")

try:
    jobs_df = scrape_jobs(
        site_name=SITES,
        search_term=SEARCH_TERM,
        location=LOCATION,
        results_wanted=RESULTS_WANTED,
        hours_old=HOURS_OLD,
        country_indeed=COUNTRY,
        is_remote=IS_REMOTE,
        verbose=1  # 0=errors only, 1=errors+warnings, 2=all logs
    )
    
    print(f"\n✅ Found {len(jobs_df)} jobs!")
    
except Exception as e:
    print(f"\n❌ Error: {e}")
    jobs_df = pd.DataFrame()

## Results Summary

In [ ]:
if len(jobs_df) > 0:
    print(f"\n📈 Summary:")
    print(f"   Total jobs: {len(jobs_df)}")
    
    if 'site' in jobs_df.columns:
        print(f"\n📊 Jobs by source:")
        print(jobs_df['site'].value_counts().to_string())
    
    if 'job_type' in jobs_df.columns:
        print(f"\n💼 Job types:")
        print(jobs_df['job_type'].value_counts().to_string())
    
    if 'is_remote' in jobs_df.columns:
        remote_count = jobs_df['is_remote'].sum() if jobs_df['is_remote'].dtype == bool else 0
        print(f"\n🏠 Remote jobs: {remote_count}")
else:
    print("\n⚠️ No jobs found. Try adjusting your search parameters.")

## Job Listings Table

In [ ]:
if len(jobs_df) > 0:
    # Select key columns for display
    display_cols = ['title', 'company', 'location', 'site', 'job_type', 'job_url']
    if 'min_amount' in jobs_df.columns and 'max_amount' in jobs_df.columns:
        display_cols.extend(['min_amount', 'max_amount'])
    if 'date_posted' in jobs_df.columns:
        display_cols.append('date_posted')
    
    # Filter to available columns
    display_cols = [col for col in display_cols if col in jobs_df.columns]
    
    # Display table
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_colwidth', 50)
    pd.set_option('display.width', None)
    
    display(jobs_df[display_cols].head(100))
else:
    print("No jobs to display.")

## Export Results

In [ ]:
if len(jobs_df) > 0:
    # Export to CSV
    csv_filename = f"jobs_{SEARCH_TERM.replace(' ', '_')}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    jobs_df.to_csv(csv_filename, index=False, encoding='utf-8')
    print(f"✅ Exported to: {csv_filename}")
    
    # Export to JSON (for web integration)
    json_filename = f"jobs_{SEARCH_TERM.replace(' ', '_')}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    jobs_df.to_json(json_filename, orient='records', indent=2, date_format='iso')
    print(f"✅ Exported to: {json_filename}")
    
    # Display download link
    display(HTML(f"<p><a href='{csv_filename}' download>📥 Download CSV</a> | <a href='{json_filename}' download>📥 Download JSON</a></p>"))
else:
    print("No jobs to export.")

## JSON Output (for API integration)

The jobs data as JSON:

In [ ]:
if len(jobs_df) > 0:
    jobs_json = jobs_df.to_dict(orient='records')
    # Clean up the JSON (remove NaN values)
    import math
    def clean_json(obj):
        if isinstance(obj, dict):
            return {k: clean_json(v) for k, v in obj.items() if not (isinstance(v, float) and math.isnan(v))}
        elif isinstance(obj, list):
            return [clean_json(item) for item in obj]
        return obj
    
    clean_jobs = clean_json(jobs_json)
    print(f"📋 JSON output ({len(clean_jobs)} jobs):")
    print(json.dumps(clean_jobs[:5], indent=2, default=str))  # Show first 5 as preview
    print(f"\n... ({len(clean_jobs) - 5} more jobs)")
else:
    print("No jobs to display.")